In [ ]:
!pip install -q pyannote.audio
!pip install -q openai-whisper
!pip install -q pydub
!pip install -q ffmpeg-python
!pip install -q fastapi uvicorn nest-asyncio pyngrok python-multipart
!apt-get -qq install ffmpeg

In [ ]:
import os
import json
import shutil
import whisper

from pydub import AudioSegment

from pyannote.audio import Pipeline
from huggingface_hub import login

from fastapi import FastAPI, UploadFile, File

import nest_asyncio
import uvicorn
from pyngrok import ngrok

In [ ]:
HF_TOKEN = "youtoken"

login(HF_TOKEN)

In [ ]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

model = whisper.load_model("base")

In [ ]:
app = FastAPI()

os.makedirs("uploads", exist_ok=True)

In [ ]:
@app.get("/")
def home():
    return {
        "message": "Speaker Diarization API Running"
    }

In [ ]:
@app.post("/diarize")
async def diarize(file: UploadFile = File(...)):

    filepath = os.path.join("uploads", file.filename)

    with open(filepath, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    print("file recieved")

    output = pipeline(filepath)
    print("diarization complete")

    diarization = output.speaker_diarization
    print("Detected speakers:\n")

    for turn, _, speaker in diarization.itertracks(yield_label=True):
        print(
            f"{speaker} : {turn.start:.2f} --> {turn.end:.2f}"
        )

    segments = []

    for turn, _, speaker in diarization.itertracks(yield_label=True):

        segments.append({
            "speaker": speaker,
            "start": turn.start,
            "end": turn.end
        })

    audio = AudioSegment.from_file(filepath)

    final_transcript = []

    for i, seg in enumerate(segments):
        print(f"Processing segment {i+1}/{len(segments)}")

        clip = audio[
            int(seg["start"]*1000):
            int(seg["end"]*1000)
        ]

        filename = f"segment_{i}.wav"

        clip.export(filename, format="wav")

        result = model.transcribe(filename)

        print(f"Finished segment {i+1}")

        final_transcript.append({
            "speaker": seg["speaker"],
            "start": seg["start"],
            "end": seg["end"],
            "text": result["text"].strip()
        })

        os.remove(filename)

    full_transcript = " ".join(
      segment["text"] for segment in final_transcript
    )
    os.remove(filepath)

    return {
      "transcript": full_transcript,
      "speaker_transcript": final_transcript
    }

In [ ]:
nest_asyncio.apply()

In [ ]:
ngrok.set_auth_token("ngrok_token")

In [ ]:
import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

In [ ]:
public_url = ngrok.connect(8000)

print(public_url.public_url)